### **Demo quá trình data indexing và tìm kiếm**

Notebook chạy toàn bộ luồng trên **một văn bản**: đọc văn bản → chia parent / child chunk → tạo vector → lưu vào MongoDB và Qdrant → hybrid search.

Dữ liệu được ghi vào kho test riêng (Qdrant `law_collection_test`, MongoDB `law_test`), không đụng vào kho thật `law_collection` / `law` của chatbot.

#### **Thiết lập môi trường**

- Dùng chung `.env` và code của backend (`App/backend/src`)
- Đổi collection Qdrant và database MongoDB sang kho test **trước khi** import code backend (các module đọc biến môi trường lúc import)

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# The notebook can run from the repo root or from a subfolder
ROOT = Path.cwd() if (Path.cwd() / "App").exists() else Path.cwd().parent
BACKEND_DIR = ROOT / "App" / "backend" / "src"
DATA_DIR = ROOT / "Data"

load_dotenv(BACKEND_DIR / ".env")
sys.path.insert(0, str(BACKEND_DIR))

# Separate test store: never touch the chatbot's real collection / database
os.environ["COLLECTION_NAME"] = "law_collection_test"
os.environ["LAW_DOC_DB"] = "law_test"

In [2]:
from qdrant_client import models
from langchain.schema import Document

from search_document_v2.custom_retriever import init_retriever
from search_document_v2.vector_store import COLLECTION_NAME

print("Qdrant collection:", COLLECTION_NAME)

/Users/duongvu/Projects/law_Q_A_system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qdrant collection: law_collection_test


#### **Khởi tạo module thực hiện quá trình data indexing**

`init_retriever()` tạo bộ chia parent / child chunk, mô hình embedding BGE-M3 (dense) và BM25 (sparse), kết nối MongoDB và Qdrant.

In [3]:
retriever = init_retriever()

# Start from an empty test store
retriever.docstore.collection.delete_many({})
retriever.vectorstore.client.delete(COLLECTION_NAME, points_selector=models.Filter(must=[]))

Pinged your deployment. You successfully connected to MongoDB!


UpdateResult(operation_id=7, status=<UpdateStatus.COMPLETED: 'completed'>)

#### **Văn bản pháp luật**

Văn bản được thu thập từ [Cơ sở dữ liệu quốc gia về văn bản pháp luật](https://vbpl.vn) bằng `Core/crawl_law_docs.ipynb` (tải danh mục, tải toàn văn, bóc tách HTML) và lưu ở `Data/doc/law_corpus.json`. Ở đây lấy một văn bản ngắn: Nghị định 45/2024/NĐ-CP.

In [4]:
import json

DOC_ID = "167276"  # Nghị định 45/2024/NĐ-CP (~3.700 words)

with open(DATA_DIR / "doc" / "law_corpus.json", "r", encoding="utf-8") as file:
    law = next(law for law in json.load(file) if law["metadata"]["id"] == DOC_ID)

doc = Document(page_content=law["data"], metadata=law["metadata"], id=DOC_ID)
print(doc.metadata["title"], "|", doc.metadata["status"], "|", len(doc.page_content.split()), "words")
print(doc.page_content[:1000])

Nghị định 45/2024/NĐ-CP | Còn hiệu lực | 3738 words
CHÍNH PHỦ 
 _________ 
 Số: 45/2024/NĐ-CP 
 CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM 
 Độc lập - Tự do - Hạnh phúc 
 ______________________________________ 
 Hà Nội, ngày 26 tháng 4 năm 2024 
 NGHỊ ĐỊNH 
 Sửa đổi, bổ sung một số điều của Nghị định số 39/2019/NĐ-CP 
 ngày 10 tháng 5 năm 2019 của Chính phủ về tổ chức và hoạt động 
 của Quỹ Phát triển doanh nghiệp nhỏ và vừa 
 _________________ 
 Căn cứ Luật Tổ chức Chính phủ ngày 19 tháng 6 năm 2015; Luật sửa đổi, bổ sung một số điều của Luật Tổ chức Chính phủ và Luật Tổ chức chính quyền địa phương ngày 22 tháng 11 năm 2019; 
 Căn cứ Luật Quản lý, sử dụng vốn nhà nước đầu tư vào sản xuất, kinh doanh tại doanh nghiệp ngày 26 tháng 11 năm 2014; 
 Căn cứ Luật Ngân sách nhà nước ngày 25 tháng 6 năm 2015; 
 Căn cứ Luật Hỗ trợ doanh nghiệp nhỏ và vừa ngày 12 tháng 6 năm 2017; 
 Căn cứ Luật Các tổ chức tín dụng ngày 16 tháng 6 năm 2010; Luật sửa đổi, bổ sung một số điều của Luật Các tổ chức tín dụng

#### **Chunking**

- **Parent chunk**: chia theo Phần / Chương / Mục / Điều, tối đa 1024 từ, lưu nguyên văn vào MongoDB. Đây là đoạn được đưa cho LLM đọc.
- **Child chunk**: chia mỗi parent theo Điều / khoản / điểm, tối đa 255 từ, lưu vào Qdrant kèm vector. Đây là đơn vị được tìm kiếm; mỗi child giữ `doc_id` trỏ về parent của nó.

In [5]:
parents = retriever.parent_splitter.split_documents([doc])
children = retriever.child_splitter.split_documents(parents)
print(len(parents), "parent chunks,", len(children), "child chunks\n")

print("=== Parent 1 ===")
print(parents[1].page_content[:800])
print("\n=== Child chunks of parent 1 ===")
for child in retriever.child_splitter.split_documents([parents[1]])[:3]:
    print("-", child.page_content[:200].replace("\n", " "))

3 parent chunks, 21 child chunks

=== Parent 1 ===

 Điều 1. Sửa đổi, bổ sung một số điều của Nghị định số 39/2019/NĐ-CP ngày 10 tháng 5 năm 2019 của Chính phủ về tổ chức và hoạt động của Quỹ Phát triển doanh nghiệp nhỏ và vừa 
 1. Sửa đổi khoản 1, khoản 2 Điều 4 như sau: 
 “1. “Người quản lý Quỹ” là người giữ chức danh, chức vụ Chủ tịch Hội đồng thành viên, thành viên Hội đồng thành viên, Giám đốc, Phó Giám đốc. 
 2. “Người lao động của Quỹ” là người làm việc cho Quỹ theo thỏa thuận, được trả lương và chịu sự quản lý, điều hành, giám sát của Quỹ theo quy định pháp luật về lao động nhưng không giữ các chức danh, chức vụ quy định tại khoản 1 Điều này.”. 
 2. Sửa đổi khoản 2 Điều 6 như sau: 
 “2. Ban kiểm soát;”. 
 3. Sửa đổi, bổ sung khoản 4, khoản 5, khoản 6, khoản 7 Điều 7 như sau: 
 a) Sửa đổi điểm a, điểm d, điểm đ và bổ sung điểm i, đi

=== Child chunks of parent 1 ===
- Điều 1. Sửa đổi, bổ sung một số điều của Nghị định số 39/2019/NĐ-CP ngày 10 tháng 5 năm 2019 của Chính phủ về tổ

#### **Data indexing**

- Tạo vector dense (BGE-M3) và vector sparse (BM25) cho các child chunk
- Lưu parent chunk vào MongoDB, child chunk kèm các vector vào Qdrant

In [6]:
retriever.add_documents([doc])

print("Qdrant points (child):", retriever.vectorstore.client.count(COLLECTION_NAME).count)
print("MongoDB docs (parent):", retriever.docstore.collection.count_documents({}))

Qdrant points (child): 21
MongoDB docs (parent): 3


#### **Hybrid search**

Câu hỏi được tìm đồng thời bằng vector dense và BM25 trên child chunk, gộp bằng RRF, rồi lấy các parent chứa child tìm được.

In [7]:
query = "Quỹ phát triển doanh nghiệp nhỏ và vừa cho vay với lãi suất như thế nào?"

parent_docs, child_hits = retriever.invoke(query)

print("=== Child chunks found ===")
for child, score in child_hits[:5]:
    print(f"{score:.3f} | {child.metadata['doc_id']} | {child.page_content[:150]}".replace("\n", " "))

print("\n=== Parent chunks (passed to rerank and the LLM) ===")
for parent_id, parent in list(parent_docs.items())[:3]:
    print(parent_id, "|", parent.page_content[:300].replace("\n", " "), "\n")

=== Child chunks found ===
0.917 | 167276_1 | 8. Bổ sung khoản 4 Điều 15 như sau:   “4. Doanh nghiệp nhỏ và vừa vay vốn của Quỹ đã hoàn trả hết nợ gốc và lãi tiền vay đầy đủ, đúng thời hạn đã thỏa
0.850 | 167276_1 | 10. Sửa đổi, bổ sung Điều 22 như sau:   a) Sửa đổi khoản 2 như sau:   “2. Ngân hàng áp dụng quy định về hoạt động cho vay của tổ chức tín dụng, chi nh
0.725 | 167276_1 | 11. Sửa đổi khoản 2, khoản 3 Điều 23 như sau:   a) Sửa đổi điểm a khoản 2 như sau:   “a) Đáp ứng quy định tại điểm c khoản 1, điểm a và điểm b khoản 2
0.717 | 167276_0 | CHÍNH PHỦ   _________   Số: 45/2024/NĐ-CP   CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM   Độc lập - Tự do - Hạnh phúc   ______________________________________ 
0.708 | 167276_1 | Điều 1. Sửa đổi, bổ sung một số điều của Nghị định số 39/2019/NĐ-CP ngày 10 tháng 5 năm 2019 của Chính phủ về tổ chức và hoạt động của Quỹ Phát triển 

=== Parent chunks (passed to rerank and the LLM) ===
167276_0 | CHÍNH PHỦ   _________   Số: 45/2024/NĐ-CP   CỘNG HÒA XÃ HỘ